In [1]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 06 - Generate and rank personalized replies

This notebook compares neutral generation, few-shot examples, and a profile-derived
instruction. The pilot is capped at 20 held-out items. The 100-item full run stays
disabled until the pilot has been reviewed and approved.

In [2]:
import os
import time
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from src.benchmark.evaluate import DEFAULT_FEATURES, StyleSpace
from src.data.splits import make_splits
from src.features.extractor import extract_message, extract_frame
from src.generation.eval_items import select_eval_items
from src.generation.generate import CONDITIONS
from src.generation.nim_client import NimClient
from src.generation.run_eval import GenerationEvaluator
from src.ranking.lgbm_ranker import train_ranker
from src.ranking.train_data import build_training_lineups

load_dotenv(ROOT / ".env")
DATA_PATH = ROOT / "data" / "processed" / "enron_messages.csv"
FINDINGS_DIR = ROOT / "docs" / "findings"
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)
N_CANDIDATES = 5
TEMPERATURE = 0.6
PILOT_SIZE = 20
FULL_SIZE = 100
RUN_FULL = False
SEED = 0

if not os.getenv("NIM_MODEL"):
    raise RuntimeError("NIM_MODEL must be configured in the local environment or .env.")
client = NimClient(requests_per_minute=30, timeout_s=120)
print("Configured model:", client.model)

Configured model: nvidia/nemotron-3-super-120b-a12b


In [3]:
df = pd.read_csv(DATA_PATH)
eligible_items = select_eval_items(df)
pilot_items = select_eval_items(df, max_items=PILOT_SIZE, seed=SEED)
full_items = select_eval_items(df, max_items=FULL_SIZE, seed=SEED)
print("Qualified held-out items:", len(eligible_items))
print("Pilot items selected:", len(pilot_items))
print("Full-run items selected:", len(full_items))

Qualified held-out items: 2803
Pilot items selected: 20
Full-run items selected: 100


In [4]:
space = StyleSpace(df, DEFAULT_FEATURES)
train_df, train_lineups = build_training_lineups(df, n_neg=9, history_frac=0.7, seed=SEED)
ranker = train_ranker(space, train_lineups, m=10, k=10.0, seed=SEED, n_estimators=60)
print("Train users:", train_df["user_id"].nunique())
print("Training lineups:", len(train_lineups))

def run_generation(items, output_prefix):
    runner = GenerationEvaluator(df, ranker, client, max_tokens=2500)
    return runner.run(
        items,
        output_prefix=output_prefix,
        checkpoint_every=10,
        resume=True,
    )


def call_stats(calls):
    rows = []
    for condition in CONDITIONS:
        group = calls[calls["condition"].eq(condition)] if not calls.empty else calls
        call_count = len(group)
        successful = group[~group["failed"]] if call_count else group
        latencies = group["latency_s"].dropna().to_numpy(dtype=float) if call_count else np.array([])
        rows.append({
            "condition": condition,
            "calls": call_count,
            "cached_calls": int(group["cached"].sum()) if call_count else 0,
            "mean_latency_s": float(np.mean(latencies)) if len(latencies) else np.nan,
            "p95_latency_s": float(np.percentile(latencies, 95)) if len(latencies) else np.nan,
            "prompt_tokens": int(group["prompt_tokens"].fillna(0).sum()) if call_count else 0,
            "completion_tokens": int(group["completion_tokens"].fillna(0).sum()) if call_count else 0,
            "failure_rate": float(group["failed"].mean()) if call_count else np.nan,
            "parse_failure_rate": (
                float(successful["parse_failure"].mean()) if len(successful) else np.nan
            ),
        })
    return pd.DataFrame(rows)


def metric_summary(candidates):
    metric_names = [
        "stylometry_similarity",
        "normalized_edit_distance",
        "lexical_similarity",
        "ranker_score",
    ]
    if candidates.empty:
        return pd.DataFrame(columns=["condition", *metric_names, "candidate_count"])
    return (
        candidates.groupby("condition")[metric_names]
        .mean()
        .assign(candidate_count=candidates.groupby("condition").size())
        .reset_index()
    )


def user_level_summary(frame, group_column, metric_names, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for group_value, group in frame.groupby(group_column, sort=True):
        user_means = group.groupby("user_id")[metric_names].mean()
        values = user_means.to_numpy(dtype=float)
        users = len(values)
        if users == 0:
            continue
        boot_means = np.array([
            values[rng.integers(0, users, users)].mean(axis=0)
            for _ in range(n_boot)
        ])
        row = {group_column: group_value, "users": users}
        for index, metric in enumerate(metric_names):
            row[f"mean_{metric}"] = float(values[:, index].mean())
            row[f"ci_low_{metric}"] = float(np.percentile(boot_means[:, index], 2.5))
            row[f"ci_high_{metric}"] = float(np.percentile(boot_means[:, index], 97.5))
        rows.append(row)
    return pd.DataFrame(rows)


def selection_test(candidates):
    rng = np.random.default_rng(SEED)
    rows = []
    instruction_rows = candidates[candidates["condition"].eq("instruction")]
    for item_id, group in instruction_rows.groupby("item_id", sort=True):
        random_index = int(rng.integers(0, len(group)))
        ranker_index = int(group["ranker_score"].to_numpy().argmax())
        picked = {
            "random": group.iloc[random_index],
            "first": group.iloc[0],
            "ranker": group.iloc[ranker_index],
        }
        for picker, selected in picked.items():
            rows.append({
                "item_id": item_id,
                "user_id": selected["user_id"],
                "picker": picker,
                "candidate": selected["candidate"],
                "stylometry_similarity": selected["stylometry_similarity"],
                "normalized_edit_distance": selected["normalized_edit_distance"],
                "lexical_similarity": selected["lexical_similarity"],
                "ranker_score": selected["ranker_score"],
            })
    return pd.DataFrame(rows)


def blind_review_tables(items, candidates):
    review_rows = []
    key_rows = []
    rng = np.random.default_rng(SEED + 10)
    by_item = items.head(10).set_index("item_id")
    for item_id, item in by_item.iterrows():
        group = candidates[candidates["item_id"].eq(item_id)]
        options = []
        for condition in CONDITIONS:
            for candidate in group[group["condition"].eq(condition)]["candidate"].tolist():
                options.append((condition, candidate))
        rng.shuffle(options)
        row = {
            "item_id": item_id,
            "incoming_message": item["incoming_message"],
            "real_reply": item["real_reply"],
        }
        for index, (condition, candidate) in enumerate(options):
            option = f"Option {chr(ord('A') + index)}"
            row[option] = candidate
            key_rows.append({
                "item_id": item_id,
                "option": option,
                "condition": condition,
            })
        review_rows.append(row)
    return pd.DataFrame(review_rows), pd.DataFrame(key_rows)


def example_table(items, candidates):
    rows = []
    for item in items.head(5).itertuples(index=False):
        row = {
            "item_id": item.item_id,
            "incoming_message": item.incoming_message,
            "real_reply": item.real_reply,
        }
        for condition in CONDITIONS:
            replies = candidates[
                candidates["item_id"].eq(item.item_id)
                & candidates["condition"].eq(condition)
            ]["candidate"].tolist()
            row[condition] = "\n---\n".join(replies)
        rows.append(row)
    return pd.DataFrame(rows)

Train users: 67
Training lineups: 5620


## Pilot: 20 held-out items

This section makes one request per item and condition. It reports request failures
and partial/invalid candidate output separately instead of hiding either case.

In [5]:
pilot_calls, pilot_candidates = run_generation(pilot_items, "06_pilot")
pilot_stats = call_stats(pilot_calls)
pilot_metrics = metric_summary(pilot_candidates)
pilot_selection = selection_test(pilot_candidates)
pilot_selection_summary = (
    pilot_selection.groupby("picker")[
        ["stylometry_similarity", "normalized_edit_distance", "lexical_similarity", "ranker_score"]
    ].mean().reset_index()
    if not pilot_selection.empty
    else pd.DataFrame()
)
pilot_examples = example_table(pilot_items, pilot_candidates)
pilot_blind_review, _blind_condition_key = blind_review_tables(pilot_items, pilot_candidates)

pilot_calls.to_csv(FINDINGS_DIR / "06_pilot_call_stats_raw.csv", index=False)
pilot_stats.to_csv(FINDINGS_DIR / "06_pilot_call_stats.csv", index=False)
pilot_candidates.to_csv(FINDINGS_DIR / "06_pilot_candidate_metrics.csv", index=False)
pilot_metrics.to_csv(FINDINGS_DIR / "06_pilot_metric_summary.csv", index=False)
pilot_selection.to_csv(FINDINGS_DIR / "06_pilot_selection.csv", index=False)
pilot_selection_summary.to_csv(FINDINGS_DIR / "06_pilot_selection_summary.csv", index=False)
pilot_examples.to_csv(FINDINGS_DIR / "06_pilot_examples.csv", index=False)
pilot_blind_review.to_csv(FINDINGS_DIR / "06_pilot_blind_review.csv", index=False)

print("Pilot call and token accounting (token totals are the available cost proxy; monetary cost is not estimated):")
display(pilot_stats.round(3))
print("Pilot independent metrics and ranker score:")
display(pilot_metrics.round(3))
print("Selection test (lower edit distance is better; higher similarities are better):")
display(pilot_selection_summary.round(3))
print("Blind human-check items: real reply is labeled; generated options are shuffled and condition names are hidden.")
display(pilot_blind_review)
print("Five pilot examples, side by side:")
display(pilot_examples)

if not pilot_selection_summary.empty:
    selection_means = pilot_selection_summary.set_index("picker")
    directions = {
        "stylometry_similarity": 1,
        "normalized_edit_distance": -1,
        "lexical_similarity": 1,
    }
    wins = 0
    for metric, direction in directions.items():
        delta = direction * (
            selection_means.loc["ranker", metric] - selection_means.loc["random", metric]
        )
        wins += int(delta > 0)
    if wins == 0:
        print("Pilot selection result: the ranker pick was not better than random on any independent metric.")
    else:
        print(f"Pilot selection result: the ranker pick was better than random on {wins} of 3 independent metrics.")

item 1 of 20, calls so far 60, elapsed 0.0s (complete; reused results)


item 2 of 20, calls so far 60, elapsed 0.0s (complete; reused results)


item 3 of 20, calls so far 60, elapsed 0.0s (complete; reused results)


item 4 of 20, calls so far 63, elapsed 61.9s


item 5 of 20, calls so far 66, elapsed 79.6s


...

Blind human-check items: real reply is labeled; generated options are shuffled and condition names are hidden.


,item_id,incoming_message,real_reply,Option A,Option B,Option C,Option D,Option E,Option F,Option G,Option H,Option I,Option J,Option K,Option L,Option M,Option N,Option O
0,887be59175,From: \tErin Richardson <erichardson@sarofim.c...,you kept rolling back over and laying into me....,"I was just trying to help you relax, not provo...","look, i’m not trying to be an attitude. if i s...",i’m sorry if i seemed pushy; just wanted us al...,i hear you; what felt cocky to you?,"I understand you feel that way, but I was mere...","my goal was to help, not annoy.","Hey, I guess we both need a break from the str...","Thanks for sharing your feelings, Erin. I'll t...",my bad if i seemed cocky. i was just being hon...,could you point out what sounded like attitude?,"hey erin, i’m sorry if i came off cocky. i was...",i get that you’re frustrated. i wasn’t trying ...,"I'm sorry if I came across that way, Erin. I d...",i respect your need for sleep; sorry for inter...,i hear you. i’ll back off and let you get to s...
1,a7c0b4f463,From: \tccates@mail.lbjs.com@ENRON [mailto:IMC...,i went to denver east. have you heard of it? ...,i attended denver south high. i enjoy the chal...,"i went to denver central high. enron's okay, i...","denver north high for me. enron's fine, just t...","hey cori, i went to east high in denver. i lik...",denver school was south high. enron work is ok...,i attended north denver high. working at enron...,"Denver school: South High. Enron's okay, I'm l...",I went to East High in Denver. Working at Enro...,i went to denver's west campus. i like enron. ...,i went to denver west high. working at enron i...,"Hey Cori, I attended West High in Denver. I li...",I went to Denver's Central High. Enron's cultu...,my denver school was east high. i like enron's...,I graduated from North Denver High. Working at...,"i went to west high in denver. i enjoy enron, ..."
2,ba740b73b3,"From: \tKeiser, Kam Sent:\tTuesday, November 2...","Kam,\n\nThanks so much for your help!\n\nKim","Hey Kam,\n\nThanks for letting me know about t...","Hi Kam,\n\nGlad to hear the ladwap issue is re...","Kam,\n\nThank you for letting me know. Please ...",thanks kam! the ladwap correction sounds great...,"Thanks, Kam.\n\nKim",thanks for the update kam! i appreciate the fi...,"Kam,\n\nGot it, thanks.\n\nKim","""thanks for the heads‑up, kam! ladwap billing ...",Appreciate the update. Please keep me informed...,Thanks for the heads‑up. Feel free to reach ou...,"Thanks for letting me know, Kam. I'll review t...","Thanks, Kam. I'll check the updated records an...","Kam,\n\nGlad to hear it’s resolved. Do you nee...","Got it, thanks. Let me know if you need anythi...","Hey Kam,\n\nAppreciate the heads‑up on the LAD..."
3,91499d0c9d,"From: \t""Dooley, Sylvia V."" <DOOLEYS@coned.com...",please print - thanks,"Thank you for sending the redlined schedule, S...","Received the redlined schedule, Sylvia. I’ll d...","Appreciate the updated document, Sylvia. I’ll ...","Thanks, Sylvia. I’ve received the redlined sch...","Sylvia, thanks for the cleaned‑up redlined ISD...",Thanks Sylvia for sending the cleaned‑up redli...,"Hi Sylvia,\nI appreciate the updated redlined ...",Thanks for the document. I’ll work on the CSA ...,"Thanks for the document, Sylvia.\nI’ll review ...","Sylvia,\nThank you for the redlined ISDA sched...","Received the attached schedule, Sylvia. Looks ...","Sylvia, thanks for the cleaned‑up version. I’l...","Got the file, Sylvia. Thanks for the cleanup—t...","Sylvia, thanks for sending the redlined schedu...",I appreciate the updated redline. Could you hi...
4,68dfa3c196,"From: Vann, Suzanne Sent: Wednesday, December ...",If I wasn't going to phx tomarrow I would say ...,"Hi Suzanne,\n\nThanks for forwarding the Beliz...","Wow, under $575 for airfare and five nights is...","suzanne, excited about the belice trip! do we ...","Hi Suzanne,\n\nThanks for the tip. Could you a...","I’m unable to leave tomorrow, but keep me in m...",That sounds like a fantastic deal—h

Five pilot examples, side by side:


,item_id,incoming_message,real_reply,neutral,fewshot,instruction
0,887be59175,From: \tErin Richardson <erichardson@sarofim.c...,you kept rolling back over and laying into me....,"I'm sorry if I came across that way, Erin. I d...","hey erin, i’m sorry if i came off cocky. i was...",i’m sorry if i seemed pushy; just wanted us al...
1,a7c0b4f463,From: \tccates@mail.lbjs.com@ENRON [mailto:IMC...,i went to denver east. have you heard of it? ...,I went to East High in Denver. Working at Enro...,"hey cori, i went to east high in denver. i lik...","i went to west high in denver. i enjoy enron, ..."
2,ba740b73b3,"From: \tKeiser, Kam Sent:\tTuesday, November 2...","Kam,\n\nThanks so much for your help!\n\nKim","Thanks for letting me know, Kam. I'll review t...","Kam,\n\nThank you for letting me know. Please ...",thanks for the update kam! i appreciate the fi...
3,91499d0c9d,"From: \t""Dooley, Sylvia V."" <DOOLEYS@coned.com...",please print - thanks,"Thanks, Sylvia. I’ve received the redlined sch...",Thanks Sylvia for sending the cleaned‑up redli...,"Thank you for sending the redlined schedule, S..."
4,68dfa3c196,"From: Vann, Suzanne Sent: Wednesday, December ...",If I wasn't going to phx tomarrow I would say ...,That sounds like a fantastic deal—how do I go ...,"Hi Suzanne,\n\nThanks for forwarding the Beliz...","hey suzanne, that belize deal sounds awesome! ..."


Pilot selection result: the ranker pick was better than random on 2 of 3 independent metrics.


## Full run: 100 held-out items

Keep the full 100-item run gated until a fresh pilot has been approved.

In [ ]:
if RUN_FULL:
    full_calls, full_candidates = run_generation(full_items, "06_full")
    full_stats = call_stats(full_calls)
    full_metrics = metric_summary(full_candidates)
    independent_metrics = [
        "stylometry_similarity",
        "normalized_edit_distance",
        "lexical_similarity",
        "ranker_score",
    ]
    full_metrics_user_ci = user_level_summary(
        full_candidates, "condition", independent_metrics
    )
    full_selection = selection_test(full_candidates)
    full_selection_user_ci = user_level_summary(
        full_selection,
        "picker",
        ["stylometry_similarity", "normalized_edit_distance", "lexical_similarity", "ranker_score"],
    )
    full_calls.to_csv(FINDINGS_DIR / "06_full_call_stats_raw.csv", index=False)
    full_stats.to_csv(FINDINGS_DIR / "06_full_call_stats.csv", index=False)
    full_candidates.to_csv(FINDINGS_DIR / "06_full_candidate_metrics.csv", index=False)
    full_metrics.to_csv(FINDINGS_DIR / "06_full_metric_summary.csv", index=False)
    full_metrics_user_ci.to_csv(FINDINGS_DIR / "06_full_metric_user_ci.csv", index=False)
    full_selection.to_csv(FINDINGS_DIR / "06_full_selection.csv", index=False)
    full_selection_user_ci.to_csv(FINDINGS_DIR / "06_full_selection_user_ci.csv", index=False)
    print("Full-run call and token accounting:")
    display(full_stats.round(3))
    print("Full-run candidate metrics with user-level 95% confidence intervals:")
    display(full_metrics_user_ci.round(3))
    print("Full-run selection test with user-level 95% confidence intervals:")
    display(full_selection_user_ci.round(3))
else:
    print("Full run is gated. Set RUN_FULL=True only after pilot approval.")